In [17]:
import glob
import json
import os
import random
import uuid
from itertools import product

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import stft, welch
from scipy.stats import entropy, norm

# Scikit-Learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras & PyTorch
import tensorflow as tf
from tensorflow.keras import Model, callbacks, layers, models
import torch

# ---------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP
# ---------------------------------------------------------------------------
print("Script execution started...")
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Set seeds for Python, NumPy, TensorFlow, and PyTorch
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print(f"Reproducibility settings locked with SEED: {SEED}")

# ---------------------------------------------------------------------------
# HARDWARE ACCELERATION CHECK
# ---------------------------------------------------------------------------
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

Script execution started...
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [18]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [19]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [20]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"
    
    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [21]:
# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_3839/1188185601.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [22]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [23]:
import mne
import numpy as np

def get_eeg_signal(
    sub_id, 
    task="walk", 
    band="full",
    target_sfreq=256,
    duration=2.0, 
    notch_freq=50.0, 
    reject_threshold=0.00028,
    noise_db=10,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, segments EEG data across all channels,
    and optionally adds Gaussian noise based on a target Signal-to-Noise Ratio (SNR in dB).

    Parameters:
    -----------
    noise_db : float or None
        Target Signal-to-Noise Ratio (SNR) in dB.
        For example, noise_db=20 adds noise such that SNR = 20 dB.
        If None, no noise is added.

    Returns:
    --------
    float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }
    
    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")
        
    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & keep full EEG channels
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. SEGMENTATION (Epoching full continuous signal)
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw, 
        events=events, 
        tmin=0, 
        tmax=duration - (1 / target_sfreq), 
        baseline=None, 
        reject=reject_criteria,
        preload=True, 
        verbose=False
    )

    # Extract array and cast to float32
    signal = epochs.get_data().astype(np.float32)

    # 8. ADD GAUSSIAN NOISE (SNR in dB)
    if noise_db is not None:
        # Calculate signal power across all dimensions
        signal_power = np.mean(signal ** 2)
        
        if signal_power > 0:
            # Convert target SNR from dB to linear scale: SNR_linear = 10^(SNR_dB / 10)
            snr_linear = 10.0 ** (noise_db / 10.0)
            
            # Calculate required noise power: Noise_Power = Signal_Power / SNR_linear
            noise_power = signal_power / snr_linear
            noise_std = np.sqrt(noise_power)
            
            # Generate zero-mean Gaussian noise with calculated std
            gaussian_noise = np.random.normal(loc=0.0, scale=noise_std, size=signal.shape).astype(np.float32)
            
            # Add noise to signal
            signal = signal + gaussian_noise

    return signal

In [24]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [25]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [26]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []

    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids

    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)

            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)

        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")

    return X_hc, X_pd


In [27]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]

    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)

    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active

        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break

        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0

                available = maj_counts[i] - allocations[i]
                take = min(share, available)

                allocations[i] += take
                remaining_target -= take

                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])

    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [28]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [29]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential

class LearnableDilatedBlock(layers.Layer):
    """
    MiniRocket-inspired Multi-Dilation Block with Trainable Weights.
    Instead of random frozen kernels, these adapt specifically to EEG frequency and shape motifs.
    """
    def __init__(self, filters, **kwargs):
        super(LearnableDilatedBlock, self).__init__(**kwargs)
        # Multi-scale temporal receptive fields using MiniRocket's standard dilation philosophy
        self.conv_d1 = layers.Conv1D(filters // 4, kernel_size=3, dilation_rate=1, padding='same')
        self.conv_d2 = layers.Conv1D(filters // 4, kernel_size=3, dilation_rate=2, padding='same')
        self.conv_d4 = layers.Conv1D(filters // 4, kernel_size=3, dilation_rate=4, padding='same')
        self.conv_d8 = layers.Conv1D(filters - 3 * (filters // 4), kernel_size=3, dilation_rate=8, padding='same')
        
        self.fusion = layers.Conv1D(filters, kernel_size=1, activation='gelu')
        self.norm = layers.LayerNormalization()

    def call(self, x, training=None):
        f1 = self.conv_d1(x)
        f2 = self.conv_d2(x)
        f4 = self.conv_d4(x)
        f8 = self.conv_d8(x)
        concat = tf.concat([f1, f2, f4, f8], axis=-1)
        out = self.fusion(concat)
        return self.norm(x + out) # Residual connection


class ChannelAttention(layers.Layer):
    """
    Weighs the importance of different EEG channels dynamically.
    """
    def __init__(self, channels, reduction=8, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.global_pool = layers.GlobalAveragePooling1D()
        self.fc1 = layers.Dense(channels // reduction, activation='gelu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')

    def call(self, x):
        # x shape: (Batch, Time, Channels)
        b, t, c = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        y = self.global_pool(x)
        y = self.fc1(y)
        y = self.fc2(y)
        y = tf.reshape(y, (b, 1, c))
        return x * y


class AttentionalRocketEEG(Model):
    """
    Enhanced MiniRocket-derived model with Trainable Dilations, Channel Attention,
    and Temporal Self-Attention for HC vs. PD Classification.
    """
    def __init__(self, in_channels=16, filters=64, dropout_rate=0.3, **kwargs):
        super(AttentionalRocketEEG, self).__init__(**kwargs)
        
        self.permute = layers.Permute((2, 1)) # (N, Channels, Time) -> (N, Time, Channels)
        
        # Initial projection
        self.input_proj = layers.Dense(filters, activation='gelu')
        
        # MiniRocket-inspired Trainable Multi-Dilation Layers
        self.rocket_block1 = LearnableDilatedBlock(filters)
        self.rocket_block2 = LearnableDilatedBlock(filters)
        
        # Channel & Temporal Attention
        self.channel_attn = ChannelAttention(filters)
        self.temporal_attn = layers.MultiHeadAttention(num_heads=4, key_dim=filters // 4)
        self.attn_norm = layers.LayerNormalization()
        
        # Classifier Head
        self.classifier_seq = Sequential([
            layers.Dense(128, activation='gelu'),
            layers.LayerNormalization(),
            layers.Dropout(dropout_rate),
            layers.Dense(64, activation='gelu'),
            layers.Dropout(dropout_rate),
            layers.Dense(1, activation='sigmoid')
        ])
        
    def call(self, inputs, training=None):
        # inputs shape: (N, Channels, Time)
        x = self.permute(inputs) # -> (N, Time, Channels)
        
        # 1. Feature projection
        x = self.input_proj(x)
        
        # 2. Multi-scale temporal sketching (Trainable MiniRocket style)
        x = self.rocket_block1(x, training=training)
        x = self.rocket_block2(x, training=training)
        
        # 3. Apply Channel Attention (focus on key brain regions)
        x = self.channel_attn(x)
        
        # 4. Apply Temporal Self-Attention (fixes MiniRocket's sequence blindness during walking)
        attn_out = self.temporal_attn(x, x, training=training)
        x = self.attn_norm(x + attn_out)
        
        # 5. Global Aggregation (Combining Mean & Max pooling over time)
        mean_pool = tf.reduce_mean(x, axis=1)
        max_pool = tf.reduce_max(x, axis=1)
        pooled = tf.concat([mean_pool, max_pool], axis=-1)
        
        # 6. Classification Output
        return self.classifier_seq(pooled, training=training)

In [30]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import product

def format_eeg_minirocket(data_array):
    """
    Ensures EEG arrays are formatted as 3D tensors: (n_cases, n_channels, n_timepoints)
    """
    arr = np.asarray(data_array, dtype=np.float32)

    if arr.ndim == 2:
        return np.expand_dims(arr, axis=0)
    elif arr.ndim == 3:
        return arr
    elif arr.ndim == 4:
        if arr.shape[-1] == 1:
            return np.squeeze(arr, axis=-1)
        elif arr.shape[1] == 1:
            return np.squeeze(arr, axis=1)
        return arr[:, :, :, 0]
    else:
        raise ValueError(f"Unexpected array dimension for model: {arr.ndim} (shape: {arr.shape})")


# ==============================================================================
# 3. SUBJECT-LEVEL NESTED CROSS-VALIDATION PIPELINE
# ==============================================================================

def run_subject_level_mc_cv_optimized(X_healthy, X_pd, SEED=42, epochs=50, batch_size=16):
    """
    Nested cross-validation pipeline using AttentionalRocketEEG.
    Fully synchronized with subject-level splitting, dynamic thresholding, and metric tracking.
    """
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # ==========================================
    # Hyperparameter Search Space
    # ==========================================
    param_grid = {
        'filters': [64],
        'dropout_rate': [0.3],
        'lr': [1e-3]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    # ==========================================
    # OUTER CROSS-VALIDATION LOOP
    # ==========================================
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 (ATTENTIONAL ROCKET) ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None
        best_threshold = 75

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        # ==========================================
        # INNER CV HYPERPARAMETER SELECTION
        # ==========================================
        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []

            for inner_fold in range(3):
                tf.keras.backend.clear_session()
                
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # 1. Balance classes for inner training (Subject-Level)
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_tr = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_tr = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

                X_tr = format_eeg_minirocket(X_tr)

                # Shuffle training epochs
                rng = np.random.RandomState(SEED + inner_fold)
                shuffle_idx = rng.permutation(len(X_tr))
                X_tr = X_tr[shuffle_idx]
                y_tr = y_tr[shuffle_idx]

                # Instantiate and compile model
                inner_model = AttentionalRocketEEG(
                    filters=params['filters'], 
                    dropout_rate=params['dropout_rate']
                )
                
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )
                
                # Fit Inner Model
                inner_model.fit(
                    X_tr, y_tr, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    verbose=0
                )

                # Inner Validation Threshold Tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_subject_ratios = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = format_eeg_minirocket(sub)
                    if sub_array.shape[0] == 0:
                        continue

                    epoch_probs = inner_model.predict(sub_array, verbose=0).ravel()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t

                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))

        print(f">> Best Rocket Grid Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")

        # ==========================================
        # OUTER TRAINING & FINAL FIT
        # ==========================================
        tf.keras.backend.clear_session()
        
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_tr_f = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
        y_tr_f = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

        X_tr_f = format_eeg_minirocket(X_tr_f)
        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_tr_f))
        X_tr_f, y_tr_f = X_tr_f[shuffle_idx_final], y_tr_f[shuffle_idx_final]

        final_model = AttentionalRocketEEG(
            filters=best_params['filters'], 
            dropout_rate=best_params['dropout_rate']
        )
        
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        # Track Training Time
        start_time = time.time()
        final_model.fit(
            X_tr_f, y_tr_f, 
            epochs=epochs, 
            batch_size=batch_size, 
            verbose=0
        )
        training_time = time.time() - start_time

        # Extract precise total learnable parameter count
        model_params_count = final_model.count_params()

        # ==========================================
        # OUTER TEST SET EVALUATION
        # ==========================================
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        min_t, max_t = thresholds[0], thresholds[-1]
        
        # Track Inference Time
        start_inf = time.time()
        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = format_eeg_minirocket(sub)
            if sub_array.shape[0] == 0:
                continue

            epoch_probs = final_model.predict(sub_array, verbose=0).ravel()
            pct_pd = float(np.mean(epoch_probs) * 100)

            vote_thresholds = [
                max(min_t, best_threshold - 5),
                best_threshold,
                min(max_t, best_threshold + 5)
            ]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1
        inference_time = time.time() - start_inf

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}",
            'Training Time (s)': f"{training_time:.2f}",
            'Inference Time (s)': f"{inference_time:.2f}",
            'Model Parameters': f"{model_params_count:,}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}% | Train Time: {training_time:.2f}s | Inf Time: {inference_time:.2f}s | Params: {model_params_count:,}")

    # ==========================================
    # FINAL SUMMARY REPORT
    # ==========================================
    summary_df = pd.DataFrame(fold_summary_records)
    total_hc_correct = sum(int(x.split('/')[0]) for x in summary_df['Healthy Correct'])
    total_hc_subjects = sum(int(x.split('/')[1]) for x in summary_df['Healthy Correct'])
    
    total_pd_correct = sum(int(x.split('/')[0]) for x in summary_df['PD Correct'])
    total_pd_subjects = sum(int(x.split('/')[1]) for x in summary_df['PD Correct'])
    
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    avg_train_time = summary_df['Training Time (s)'].astype(float).mean()
    avg_inf_time = summary_df['Inference Time (s)'].astype(float).mean()
    avg_params = summary_df['Model Parameters'].str.replace(',', '').astype(int).mean()

    print(f"\n========================================")
    print(f"========== FINAL EXPERIMENT SUMMARY =====")
    print(f"========================================")
    print(f"Healthy Controls Correct: {total_hc_correct} / {total_hc_subjects} ({total_hc_correct/total_hc_subjects*100:.2f}%)")
    print(f"Parkinson's Disease (PD) Correct: {total_pd_correct} / {total_pd_subjects} ({total_pd_correct/total_pd_subjects*100:.2f}%)")
    print(f"Total Combined Correct: {total_correct} / {total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy (Attentional Rocket): {overall_acc:.2f}%")
    print(f"----------------------------------------")
    print(f"Average Training Time: {avg_train_time:.2f} seconds")
    print(f"Average Inference Time: {avg_inf_time:.2f} seconds")
    print(f"Average Model Parameters: {int(avg_params):,}")
    print(f"========================================")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [31]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3839/117235121.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_3839/117235121.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3839/117235121.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_3839/117235121.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

2026-09-15 13:00:10.190146: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-15 13:04:44.094867: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Rocket Grid Selected: {'filters': 64, 'dropout_rate': 0.3, 'lr': 0.001} | Threshold: 75% (Inner Acc: 0.8108)


2026-09-15 13:14:54.255723: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-15 13:21:52.528509: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 3/5 | PD: 20/23 | Acc: 82.14% | Train Time: 420.28s | Inf Time: 5.29s | Params: 80,137

========== FINAL EXPERIMENT SUMMARY =====
Healthy Controls Correct: 21 / 28 (75.00%)
Parkinson's Disease (PD) Correct: 95 / 116 (81.90%)
Total Combined Correct: 116 / 144
Overall Nested Cross-Validation Accuracy (Attentional Rocket): 80.56%
----------------------------------------
Average Training Time: 408.07 seconds
Average Inference Time: 5.44 seconds
Average Model Parameters: 80,137

--- Nested Cross-Validation Summary ---
 Fold Number                               Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Inference Time (s) Model Parameters
           1 {'filters': 64, 'dropout_rate': 0.3, 'lr': 0.001}                     80             6/6      17/24            76.67%         23/30            400.19               5.54           80,137
           2 {'filters': 64, 'dropout_rate': 0.3, 'l

2026-09-15 13:21:57.631265: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [32]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3839/117235121.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_3839/117235121.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3839/117235121.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_3839/117235121.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 3/4 | PD: 17/22 | Acc: 76.92% | Train Time: 317.57s | Inf Time: 4.95s | Params: 80,137

========== OUTER FOLD 5 / 5 (ATTENTIONAL ROCKET) ==========


2026-09-15 14:28:57.559628: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-15 14:32:24.334077: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Rocket Grid Selected: {'filters': 64, 'dropout_rate': 0.3, 'lr': 0.001} | Threshold: 65% (Inner Acc: 0.7770)


2026-09-15 14:40:34.701145: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-15 14:46:04.342763: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 2/4 | PD: 21/22 | Acc: 88.46% | Train Time: 331.26s | Inf Time: 4.88s | Params: 80,137

========== FINAL EXPERIMENT SUMMARY =====
Healthy Controls Correct: 11 / 23 (47.83%)
Parkinson's Disease (PD) Correct: 92 / 110 (83.64%)
Total Combined Correct: 103 / 133
Overall Nested Cross-Validation Accuracy (Attentional Rocket): 77.44%
----------------------------------------
Average Training Time: 313.31 seconds
Average Inference Time: 5.10 seconds
Average Model Parameters: 80,137

--- Nested Cross-Validation Summary ---
 Fold Number                               Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct Training Time (s) Inference Time (s) Model Parameters
           1 {'filters': 64, 'dropout_rate': 0.3, 'lr': 0.001}                     65             4/5      16/22            74.07%         20/27            316.13               4.84           80,137
           2 {'filters': 64, 'dropout_rate': 0.3, 'l